curl "https://api.openf1.org/v1/laps?session_key=9839&driver_number=44&lap_number<=3"

In [0]:
%python
catalog     = "workspace"
session_key =  9839
drivers      =  44
base_url    = "https://api.openf1.org/v1"

In [0]:
%python

import requests
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructType, StructField

#--1. Chamando a API ---------
resp_drivers = requests.get(
    f"{base_url}/drivers",
    params={"session_key": session_key},
    timeout =30
)
resp_drivers.raise_for_status()
drivers = [d["driver_number"] for d in resp_drivers.json()]

print(f"Pilotos encontrados na sessão {session_key}: {drivers}")

In [0]:
%python
import requests
import json
from pyspark.sql import functions as f
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Inicialização da lista
all_rows = []

# Loop pelos pilotos
for driver in drivers:
    resp = requests.get(
        f"{base_url}/laps",
        params={"session_key": session_key, "driver_number": driver},
        timeout=30
    )
    resp.raise_for_status()
    laps = resp.json()
    print(f" Piloto {driver}: {len(laps)} voltas")

    for record in laps:
        all_rows.append((
            json.dumps(record),
            driver,
        ))

# Definição do schema
schema_bronze = StructType([
    StructField("raw_payload",      StringType(), False),
    StructField("_driver_number",  IntegerType(), True),
])

# Criação do DataFrame
df_bronze = (
    spark.createDataFrame(all_rows, schema=schema_bronze)
    .withColumn("_ingested_at", f.current_timestamp())
    .withColumn("_session_key", f.lit(session_key))
)

# Escrita na tabela
(
    df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{catalog}.bronze.laps_raw")
)

print(f"\n Bronze: {df_bronze.count()} registros totais gravados")

In [0]:
drop table if exists
default.laps_raw;

In [0]:
select * from
workspace.bronze.laps_raw limit 100